# CSE428 Project — Pet Segmentation & Breed Classification

**Dataset:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) · 37 breeds · 3,680 trainval / 3,666 test images

**Models:** U-Net and Attention U-Net with a breed-classifier head on the shared encoder (trained jointly)

**Contents:** data exploration → base U-Net → Attention U-Net → results (mIoU, Dice, pixel accuracy · accuracy, precision, recall, F1)

> Training runs in **resumable checkpoint chunks**: each notebook version saves its state to the output, and the next run (or a groupmate) resumes from it — never start from scratch.

## 0. Repository sync — code comes from GitHub, no copy-paste

All project code lives in a public repo. This cell clones it (fresh session) or pulls the latest version (existing session).

In [ ]:
import os, sys

REPO_URL = "https://github.com/shahriar-abid/cse428-pets.git"
REPO_DIR = "/kaggle/working/cse428-pets"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("repo ready:", os.getcwd())

## 1. Setup

In [ ]:
import yaml
import torch

from src.utils import seed_everything, get_device

CFG = yaml.safe_load(open("configs/config.yaml"))
seed_everything(CFG["seed"])
DEVICE = get_device()
print("device:", DEVICE)
print("config:", CFG)

## 2. Dataset & Exploration

**Oxford-IIIT Pet** — 3,680 trainval + 3,666 test images across 37 breeds.

Each image ships with a **trimap**: `1` = foreground (pet), `2` = background, `3` = boundary (not classified). Per the project guidelines, boundary pixels are merged into the foreground, giving a binary mask: `background → 0`, `foreground + boundary → 1`.

Split: 90% train / 10% validation from trainval (deterministic, seeded — identical across sessions so checkpointed training resumes on the same data). The official test set is used for testing.

In [ ]:
from src.data import get_loaders

DATASETS, LOADERS = get_loaders(
    root=CFG["data"]["root"],
    img_size=CFG["data"]["img_size"],
    val_frac=CFG["data"]["val_frac"],
    seed=CFG["seed"],
    augment=CFG["data"]["augment"],
    batch_size=CFG["data"]["batch_size"],
    num_workers=CFG["data"]["num_workers"],
    download=True,
)
for name, d in DATASETS.items():
    print(f"{name}: {len(d)} samples ({len(d.classes)} classes)")

### 2.1 Images with mask overlays (3x3, required format)

In [ ]:
import matplotlib.pyplot as plt
from src.viz import plot_overlay_grid

fig = plot_overlay_grid(DATASETS["val"], figsize=(12, 12))
plt.show()

## 3. Base U-Net — segmentation + classification

*Model, training loop (with checkpoint resume) and metrics land here on day 2–3; this notebook trains in resumable checkpoint chunks.*